In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")


In [2]:
DATA_ROOT = "dataset"
IMG_SIZE = 300          # EfficientNetB2 native
BATCH_SIZE = 32
EPOCHS_HEAD = 12
EPOCHS_FINE = 15
VAL_SPLIT = 0.15
SEED = 1337


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
num_classes = len(class_names)


Found 8473 files belonging to 3 classes.
Using 7203 files for training.
Found 8473 files belonging to 3 classes.
Using 1270 files for validation.


In [4]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


In [5]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])


In [6]:
base_model = keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling="avg",
)

base_model.trainable = False


In [7]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="swish")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax",
    dtype="float32"
)(x)

model = keras.Model(inputs, outputs)


In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)


In [9]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3),
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)


Epoch 1/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 679s 3s/step - accuracy: 0.6783 - loss: 0.9610 - val_accuracy: 0.7693 - val_loss: 0.6937 - learning_rate: 3.0000e-04
Epoch 2/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 656s 3s/step - accuracy: 0.7380 - loss: 0.8108 - val_accuracy: 0.7835 - val_loss: 0.7002 - learning_rate: 3.0000e-04
Epoch 3/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 660s 3s/step - accuracy: 0.7626 - loss: 0.7395 - val_accuracy: 0.7921 - val_loss: 0.6761 - learning_rate: 3.0000e-04
Epoch 4/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 657s 3s/step - accuracy: 0.7756 - loss: 0.7091 - val_accuracy: 0.7913 - val_loss: 0.6860 - learning_rate: 3.0000e-04
Epoch 5/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 656s 3s/step - accuracy: 0.7886 - loss: 0.6817 - val_accuracy: 0.7740 - val_loss: 0.6792 - learning_rate: 3.0000e-04
Epoch 6/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 655s 3s/step - accuracy: 0.8036 - loss: 0.6459 - val_accuracy: 0.8079 - val_loss: 0.6299 - learning_rate: 9.0000e-05
Epoch 7/12
226/226 ━━━━━━━━━━━━━━━━━━━━ 652s 3s/step - acc

In [10]:
model.save("skin_cancer_model.keras")
